In [2]:
# Configuración inicial
import sys, os
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv()

# Añadir el repo clonado al PYTHONPATH
sys.path.append("./neo4j_graphrag_tutorial")

# Verificar configuración
print("✅ Variables de entorno cargadas desde .env")
print(f"   OPENAI_API_KEY: {'✓ Configurada' if os.getenv('OPENAI_API_KEY') else '✗ No encontrada'}")
print(f"   NEO4J_URI: {os.getenv('NEO4J_URI', '✗ No configurada')}")
print(f"   NEO4J_USERNAME: {os.getenv('NEO4J_USERNAME', '✗ No configurado')}")
print(f"   NEO4J_PASSWORD: {'✓ Configurada' if os.getenv('NEO4J_PASSWORD') and os.getenv('NEO4J_PASSWORD') != 'your_password_here' else '✗ No configurada o placeholder'}")

neo4j_ready = (
    os.getenv('NEO4J_URI') and 
    os.getenv('NEO4J_PASSWORD') and 
    os.getenv('NEO4J_PASSWORD') != 'Mdps2004'
)

if not neo4j_ready:
    print("\n⚠️  ADVERTENCIA: Neo4j no está configurado correctamente")
    print("   Para usar GraphRAG necesitas:")
    print("   1. Neo4j corriendo (Desktop o Aura)")
    print("   2. Configurar credenciales en .env")
    print("   3. Cargar datos con el ETL del tutorial")
else:
    print("\n✅ Neo4j configurado - listo para evaluar GraphRAG")

✅ Variables de entorno cargadas desde .env
   OPENAI_API_KEY: ✓ Configurada
   NEO4J_URI: neo4j://127.0.0.1:7687
   NEO4J_USERNAME: neo4j
   NEO4J_PASSWORD: ✓ Configurada

⚠️  ADVERTENCIA: Neo4j no está configurado correctamente
   Para usar GraphRAG necesitas:
   1. Neo4j corriendo (Desktop o Aura)
   2. Configurar credenciales en .env
   3. Cargar datos con el ETL del tutorial


# Evaluación de pipeline Neo4j RAG con evaluador universal

Este cuaderno permite definir datasets de evaluación, importar el wrapper y la función de evaluación universal, y lanzar la evaluación sobre el pipeline Neo4j (`qa_chain`).

In [3]:
# Importar el wrapper standalone y el evaluador
from graphrag_wrapper_standalone import neo4j_graphrag_wrapper_standalone as graphrag_wrapper
from rag_evaluator import evaluate_rag

print("✅ Imports cargados correctamente")
print("   - graphrag_wrapper (GraphCypherQAChain standalone)")
print("   - evaluate_rag (evaluador universal)")

✅ Imports cargados correctamente
   - graphrag_wrapper (GraphCypherQAChain standalone)
   - evaluate_rag (evaluador universal)


## Definir dataset de evaluación

Modifica aquí las preguntas y respuestas esperadas según tu caso y tus datos en Neo4j.

In [27]:
dataset_neo4j = [
    {
        "inputs": {"question": "How many employees are based in the USA?"},
        "outputs": {"answer": "There are 5 employees based in the USA."}
    },
    {
        "inputs": {"question": "How many employees work in the UK?"},
        "outputs": {"answer": "There are 4 employees who work in the UK."}
    },
    {
        "inputs": {"question": "Who is the Sales Manager?"},
        "outputs": {"answer": "The Sales Manager is Robert King."}
    },
    {
        "inputs": {"question": "How many sales representatives does the company have?"},
        "outputs": {"answer": "The company has 6 sales representatives"}
    },
    {
        "inputs": {"question": "Which employees have the title 'Sales Representative'?"},
        "outputs": {"answer": """"Respuesta: The employees with the title 'Sales Representative' are:
1. Michael Suyama in the UK
2. Margaret Peacock in the USA
3. Janet Leverling in the USA
4. Anne Dodsworth in the UK
5. Nancy Davolio in the USA
6. Robert King in the UK
"""}
    },
    {
        "inputs": {"question": "Which employees work in London?"},
        "outputs": {"answer": """The employees who work in London are:
1. Steven Buchanan - Sales Manager
2. Michael Suyama - Sales Representative
3. Anne Dodsworth - Sales Representative
4. Robert King - Sales Representative
"""}
    }
]

## Ejecutar la evaluación y visualizar resultados

Esto lanzará la evaluación sobre el pipeline Neo4j usando el dataset definido arriba.

## Test del wrapper (opcional)

Antes de ejecutar la evaluación completa, puedes probar que el wrapper funciona correctamente:

In [ ]:
# Test rápido del wrapper
print("🧪 Test del wrapper GraphRAG...")
test_input = {"question": "Which employees work in the London region?"}
print(f"Pregunta: {test_input['question']}")

try:
    result = graphrag_wrapper(test_input)
    print(f"\n✅ Wrapper ejecutado correctamente")
    print(f"Respuesta: {result['answer'][:300] if result['answer'] else 'Sin respuesta'}")
    
    if "ERROR" in result['answer']:
        print("\n⚠️  Hay un error en la respuesta. Posibles causas:")
        print("   - Neo4j no está corriendo")
        print("   - Credenciales incorrectas en .env")
        print("   - Datos no cargados en Neo4j")
        print("\nDetalles del error:")
        print(result['answer'])
except Exception as e:
    print(f"\n❌ Error ejecutando wrapper: {e}")
    import traceback
    traceback.print_exc()

🧪 Test del wrapper GraphRAG...
Pregunta: Which employees work in London?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (e:Employee)
WHERE e.city = 'London'
RETURN e;
Full Context:
[{'e': {'country': 'UK', 'lastName': 'Buchanan', 'extension': 3453, 'hireDate': '1993-10-17 00:00:00.000', 'address': '14 Garrett Hill', 'notes': 'Steven Buchanan graduated from St. Andrews University in Scotland with a BSC degree in 1976.  Upon joining the company as a sales representative in 1992 he spent 6 months in an orientation program at the Seattle office.', 'city': 'London', 'photoPath': 'http://accweb/emmployees/buchanan.bmp', 'postalCode': 'SW1 8JR', 'homePhone': '(71) 555-4848', 'photo': '0x151C2F00020000000D000E0014002100FFFFFFFF4269746D617020496D616765005061696E742E506963747572650001050000020000000700000050427275736800000000000000000020540000424D20540000000000007600000028000000C0000000DF0000000100040000000000A0530000CE0E0000D80E0000000000', 'employeeID': 5, 'title': 'Sales

In [ ]:
# Ejecutar la evaluación con el wrapper standalone
print("🚀 Iniciando evaluación del GraphRAG Neo4j...")
print(f"   Dataset: {len(dataset_neo4j)} ejemplos")
print(f"   Wrapper: GraphCypherQAChain (standalone, sin conflictos)")
print()

eval_results = evaluate_rag(
    graphrag_wrapper,  # Wrapper standalone - evita conflictos de imports
    dataset_neo4j,
    dataset_name="neo4j-graphrag-eval-v3",  # Nombre único
    project="graphrag-neo4j-evaluation"
)

print("\n✅ Evaluación completada!")
print("📊 Revisa los resultados en LangSmith: https://smith.langchain.com")

eval_results

## 🔧 Troubleshooting

Si obtienes errores, verifica:

### 1. Neo4j está corriendo
```bash
# Para Neo4j Desktop: verifica que esté iniciado
# Para Neo4j Aura: verifica la conexión
```

### 2. Credenciales correctas en `.env`
```bash
NEO4J_URI=bolt://localhost:7687  # o tu URI de Aura
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=tu_password_real  # NO "your_password_here"
```

### 3. Datos cargados en Neo4j
```bash
cd neo4j_graphrag_tutorial
python ETL_pipelines/main_etl.py
```

### 4. Dependencias instaladas
```bash
pip install langchain langchain-community langchain-openai neo4j python-dotenv
```

Si todo falla, ejecuta el script de debugging:
```bash
python debug_graphrag.py
```